In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Crypto-Arithmetic Puzzle

In this exercise we will solve the crypto-arithmetic puzzle shown in the picture below:
<img src="send-more-money.png">

The idea is that the letters 
"$\texttt{S}$", "$\texttt{E}$", "$\texttt{N}$", "$\texttt{D}$", "$\texttt{M}$", "$\texttt{O}$", "$\texttt{R}$", "$\texttt{Y}$" occurring in this puzzle
are interpreted as variables ranging over the set of decimal digits, i.e. these variables can take values in
the set $\{0,1,2,3,4,5,6,7,8,9\}$.  Then, the string "$\texttt{SEND}$" is interpreted as a decimal number,
i.e. it is interpreted as the number
$$\texttt{S} \cdot 10^3 + \texttt{E} \cdot 10^2 + \texttt{N} \cdot 10^1 + \texttt{D} \cdot 10^0.$$
The strings "$\texttt{MORE}$ and "$\texttt{MONEY}$" are interpreted similarly. To make the problem
interesting, the assumption is that different variables have different values.  Furthermore, the
digits at the beginning of a number should be different from $0$.  Then, we have to find values for the variables
"$\texttt{S}$", "$\texttt{E}$", "$\texttt{N}$", "$\texttt{D}$", "$\texttt{M}$", "$\texttt{O}$", "$\texttt{R}$", "$\texttt{Y}$" such that the formula
$$   (\texttt{S} \cdot 10^3 + \texttt{E} \cdot 10^2 + \texttt{N} \cdot 10 + \texttt{D}) 
  + (\texttt{M} \cdot 10^3 + \texttt{O} \cdot 10^2 + \texttt{R} \cdot 10 + \texttt{E})
  = \texttt{M} \cdot 10^4 + \texttt{O} \cdot 10^3 + \texttt{N} \cdot 10^2 + \texttt{E} \cdot 10 + \texttt{Y}
$$
is true.  The problem with this constraint is that it involves far too many variables.  As this constraint can only be
checked when all the variables have values assigned to them, the backtracking search would essentially
boil down to a mere brute force search.  We would have 8 variables and hence we would have to test 
$$ 10 \cdot 9 \cdot \dots \cdot 3 = 1,814,400 $$
possible assignments. While it is certainly feasible to test this number of assignments on a modern computer, we want to do better.  To this end we have to perform the addition in the figure shown above
column by column, just as it is taught in elementary school.  To be able to do this, we have to introduce <a href="https://en.wikipedia.org/wiki/Carry_(arithmetic)">carry digits</a> "$\texttt{C1}$", "$\texttt{C2}$", "$\texttt{C3}$" where $\texttt{C1}$ is the carry produced by adding 
$\texttt{D}$ and $\texttt{E}$, $\texttt{C2}$ is the carry produced by adding 
$\texttt{N}$, $\texttt{R}$ and $\texttt{C1}$, and $\texttt{C3}$ is the carry produced by adding 
$\texttt{E}$, $\texttt{O}$ and $\texttt{C2}$. 

We import the backtracking solver.  The notebook `02-Backtracking-Constraint-Solver.ipynb` defines the types `Constraint<'V>` and `CSP<'V>`, the functions `constr`, `different`, and `hasValue` that create constraints, and the function `solve`.

In [2]:
importNotebook "02-Backtracking-Constraint-Solver.ipynb"

imported 

02-Backtracking-Constraint-Solver.ipynb

The function `allDifferent` takes one argument:
- `Variables` is a list of variables.

It returns a list of constraints that express that all the variables in `Variables` have different values. 

In [3]:
let allDifferent (Variables: string list) : Constraint<'V> list =
    [ for x in Variables do
        for y in Variables do
            if x < y then different x y ]

In [4]:
(allDifferent [ "a"; "b"; "c" ] : Constraint<int> list)

[a ≠ b; a ≠ c; b ≠ c]

The function `columnConstraints` takes four arguments:
- `xs` is the list of the variables that are added in a column, including the carry from the previous column,
- `z` is the variable for the digit of the sum in this column,
- `carry` is the variable for the carry produced by this column.

It returns two constraints that specify that the last digit of the sum of `xs` is `z` and that `carry` is the carry of this sum.  In F#, the operator `/` performs integer division when applied to integers.

In [5]:
let columnConstraints (xs: string list) (z: string) (carry: string) : Constraint<int> list =
    let sum (A: Map<string, int>) = List.sum [ for x in xs -> A[x] ]
    let text = String.concat " + " xs
    [ constr (sprintf "(%s) %% 10 = %s" text z) (z :: xs) (fun A -> sum A % 10 = A[z])
      constr (sprintf "(%s) / 10 = %s" text carry) (carry :: xs) (fun A -> sum A / 10 = A[carry]) ]

In [6]:
columnConstraints [ "D"; "E" ] "Y" "C1"

[(D + E) % 10 = Y; (D + E) / 10 = C1]

The function `createCSP` takes no arguments.  It returns a *constraint satisfaction problem* that encodes the
crypto-arithmetic puzzle.  The order of the variables is chosen so that the constraints for the columns can be checked as early as possible.

In [7]:
let createCSP () : CSP<int> =
    { Variables   = [ "D"; "E"; "Y"; "C1"; "N"; "R"; "C2"; "O"; "C3"; "S"; "M" ]
      Values      = [ 0 .. 9 ]
      Constraints = allDifferent [ "D"; "E"; "Y"; "N"; "R"; "O"; "S"; "M" ]
                    @ [ constr "S ≠ 0" ["S"] (fun A -> A["S"] <> 0)
                        constr "M ≠ 0" ["M"] (fun A -> A["M"] <> 0) ]
                    @ columnConstraints [ "D"; "E" ]       "Y" "C1"
                    @ columnConstraints [ "N"; "R"; "C1" ] "E" "C2"
                    @ columnConstraints [ "E"; "O"; "C2" ] "N" "C3"
                    @ columnConstraints [ "S"; "M"; "C3" ] "O" "M" }

In [8]:
let puzzle = createCSP ()
puzzle

{ Variables = ["D"; "E"; "Y"; "C1"; "N"; "R"; "C2"; "O"; "C3"; "S"; "M"]; Values = [0; 1; 2; 3; 4; 5; 6; 7; 8; 9]; Constraints = [D ≠ E; D ≠ Y; D ≠ N; D ≠ R; D ≠ O; D ≠ S; D ≠ M; E ≠ Y; E ≠ N; E ≠ R; E ≠ O; E ≠ S; E ≠ M; N ≠ Y; N ≠ R; N ≠ O; N ≠ S; R ≠ Y; R ≠ S; O ≠ Y; O ≠ R; O ≠ S; S ≠ Y; M ≠ Y; M ≠ N; M ≠ R; M ≠ O; M ≠ S; S ≠ 0; M ≠ 0; (D + E) % 10 = Y; (D + E) / 10 = C1; (N + R + C1) % 10 = E; (N + R + C1) / 10 = C2; (E + O + C2) % 10 = N; (E + O + C2) / 10 = C3; (S + M + C3) % 10 = O; (S + M + C3) / 10 = M] }

In [9]:
#!time
let Solution = solve puzzle

Wall time: 57.7284ms

In [10]:
Solution

Some map [("C1", 1); ("C2", 1); ("C3", 0); ("D", 7); ("E", 5); ("M", 1); ("N", 6); ("O", 0); ("R", 8); ("S", 9); ("Y", 2)]

The function `printSolution` takes one argument:
- `A` is the result of the function `solve`, i.e. either `Some Solution` or `None`.

It prints the values of the letters of the puzzle.

In [11]:
let printSolution (A: Map<string, int> option) : unit =
    match A with
    | None -> printfn "no solution found"
    | Some s ->
        for v in [ "S"; "E"; "N"; "D"; "M"; "O"; "R"; "Y" ] do
            printfn "%s = %d" v s[v]

In [12]:
printSolution Solution

S

 = 

9

E

 = 

...

 = 

8

Y

 = 

2